# RECAST End-to-End LoRA Pipeline

**raw → clean → augmentation → 5-fold LoRA CV → final score (∈ [0, 1])**

Wires together the existing modules in `src/crllm/`:
- `dataset.preprocess.preprocess.run_pipeline` — cleaning
- `dataset.augmentation.augment.run_augmentation` — lexical edit + back-translation
- `pipeline.lora_cv.run_kfold` — train a fresh LoRA adapter on K-1 folds, evaluate CSR on held-out fold, repeat K times

Designed to run on Colab T4. Knobs for subsampling are at the top — defaults are chosen so the full run finishes in a few hours.

## 1. Install dependencies

In [ ]:
!pip install -q transformers peft datasets accelerate trl scikit-learn \
    langdetect emoji nltk datasketch joblib sentencepiece sacremoses python-dotenv
import nltk
nltk.download('stopwords', quiet=True)

## 2. Clone repo & set up Python path

In [ ]:
import os, sys, subprocess
REPO_URL    = "https://github.com/shreyasnat2804/stat-453-constraint-based-llm.git"
REPO_BRANCH = "feature/lora-results"
REPO_DIR    = "/content/stat-453-constraint-based-llm"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", REPO_BRANCH], check=False)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", REPO_BRANCH], check=False)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)

# Imports use `from src.crllm...` (matches augment.py's internal imports)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import torch
print("CWD :", os.getcwd())
print("Repo:", REPO_DIR)
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — Runtime > Change runtime type > GPU")

## 3. Load HuggingFace token from `.env`

`.env` is gitignored, so it will not be in the cloned repo. Place it in **one** of these locations before running this cell:

- `/content/.env` — drag-and-drop into the Colab Files panel (simplest, but lost on session end)
- `/content/drive/MyDrive/.env` — mount Drive first; persists across sessions
- `<REPO_DIR>/.env` — manually copy after cloning

Format: `HF_TOKEN=hf_xxxxxxxx` on a single line.

Falls back to Colab Secrets (`HF_TOKEN`) and `os.environ` if no `.env` is found.

In [ ]:
from pathlib import Path
from dotenv import dotenv_values
from huggingface_hub import login

ENV_CANDIDATES = [
    Path("/content/.env"),
    Path("/content/drive/MyDrive/.env"),
    Path(REPO_DIR) / ".env",
    Path.cwd() / ".env",
]

hf_token = ""
env_source = None
for p in ENV_CANDIDATES:
    if p.exists():
        vals = dotenv_values(p)
        tok = (vals.get("HF_TOKEN") or "").strip()
        if tok:
            hf_token = tok
            env_source = str(p)
            break

if not hf_token:
    try:
        from google.colab import userdata
        hf_token = (userdata.get("HF_TOKEN") or "").strip()
        if hf_token:
            env_source = "colab.userdata"
    except Exception:
        pass

if not hf_token:
    hf_token = os.environ.get("HF_TOKEN", "").strip()
    if hf_token:
        env_source = "os.environ"

if not hf_token:
    raise RuntimeError(
        "HF_TOKEN not found. Place a .env file at one of: "
        + ", ".join(str(p) for p in ENV_CANDIDATES)
        + " — or set it as a Colab Secret."
    )

login(token=hf_token)
print(f"Logged in to HuggingFace (source: {env_source}).")

## 4. Pipeline configuration

Edit these knobs to scale the run. Defaults target a single Colab T4 session.

In [ ]:
from pathlib import Path

# ── Paths ────────────────────────────────────────────────────────────────────
RAW_ZIP        = Path(REPO_DIR) / "datasets" / "recast_30k_original.jsonl.zip"
WORK_DIR       = Path("/content/pipeline_work"); WORK_DIR.mkdir(exist_ok=True)
RAW_JSONL      = WORK_DIR / "recast_30k_original.jsonl"
CLEAN_JSONL    = WORK_DIR / "recast_30k_clean.jsonl"
AUG_JSONL      = WORK_DIR / "recast_30k_augmented.jsonl"
CV_OUTPUT_DIR  = WORK_DIR / "kfold_lora"

# ── Cleaning ─────────────────────────────────────────────────────────────────
MIN_LENGTH          = 15
DEDUP_THRESHOLD     = 0.85
IMBALANCE_THRESHOLD = 0.5
N_JOBS              = -1

# ── Augmentation ─────────────────────────────────────────────────────────────
AUG_BATCH_SIZE      = 32
AUG_INTERMEDIATE_LANG = "de"
SKIP_AUGMENTATION   = False  # set True to skip back-translation (slow on T4)

# ── K-Fold LoRA ──────────────────────────────────────────────────────────────
K_FOLDS             = 5
SUBSAMPLE_N         = 5000   # cap records before CV (None = use everything; ~90k after augmentation is too slow on T4)
EVAL_PER_FOLD       = 200    # examples per fold for CSR scoring
SEED                = 42

LORA_RANK           = 8
LEARNING_RATE       = 1e-4
NUM_EPOCHS          = 1
PER_DEVICE_BS       = 8
GRAD_ACCUM_STEPS    = 4
MAX_SEQ_LENGTH      = 512
BASE_MODEL          = "meta-llama/Llama-3.2-1B-Instruct"

print("Config loaded.")
print(f"  Subsample           : {SUBSAMPLE_N}")
print(f"  K folds             : {K_FOLDS}")
print(f"  Eval per fold       : {EVAL_PER_FOLD}")
print(f"  LoRA rank / lr      : {LORA_RANK} / {LEARNING_RATE}")
print(f"  Epochs / batch      : {NUM_EPOCHS} / {PER_DEVICE_BS} (×{GRAD_ACCUM_STEPS} accum)")

## 5. Stage 1 — raw → clean

Unzip the raw RECAST-30K and run the cleaning pipeline (language filter, HTML/emoji strip, stopword removal on prompts, constraint integrity, exact + MinHash dedup).

In [ ]:
import logging, zipfile
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S", force=True)

if not RAW_JSONL.exists():
    with zipfile.ZipFile(RAW_ZIP) as z:
        z.extractall(WORK_DIR)
    # The archive may extract under a different name — pick the first .jsonl
    extracted = next((p for p in WORK_DIR.glob("*.jsonl") if "clean" not in p.name), None)
    if extracted and extracted != RAW_JSONL:
        extracted.rename(RAW_JSONL)
    print(f"Extracted: {RAW_JSONL}")
else:
    print(f"Already extracted: {RAW_JSONL}")

from src.crllm.dataset.preprocess.preprocess import run_pipeline as run_clean

if not CLEAN_JSONL.exists():
    stats = run_clean(
        input_path          = RAW_JSONL,
        output_path         = CLEAN_JSONL,
        min_length          = MIN_LENGTH,
        dedup_threshold     = DEDUP_THRESHOLD,
        imbalance_threshold = IMBALANCE_THRESHOLD,
        n_jobs              = N_JOBS,
    )
    print(f"\nClean kept: {stats['kept']:,} / {stats['total']:,}")
else:
    print(f"Reusing cached clean file: {CLEAN_JSONL}")

## 6. Stage 2 — clean → augmented

Lexical edit + back-translation. Output is roughly 3× the clean record count (original + lex + bt).

In [ ]:
from src.crllm.dataset.augmentation.augment import run_augmentation

if SKIP_AUGMENTATION:
    print("SKIP_AUGMENTATION=True — using clean dataset for CV.")
    AUG_JSONL = CLEAN_JSONL
elif not AUG_JSONL.exists():
    aug_stats = run_augmentation(
        input_path        = CLEAN_JSONL,
        output_path       = AUG_JSONL,
        seed              = SEED,
        force             = False,
        intermediate_lang = AUG_INTERMEDIATE_LANG,
        batch_size        = AUG_BATCH_SIZE,
        max_length        = MAX_SEQ_LENGTH,
    )
    print(f"\nAugmentation: {aug_stats['total_output']:,} records "
          f"(orig {aug_stats['original']:,} + lex {aug_stats['lexical_edit']:,} + bt {aug_stats['back_translation']:,})")
else:
    print(f"Reusing cached augmented file: {AUG_JSONL}")

## 7. Stage 3 — 5-fold LoRA cross-validation

For each fold:
1. Train a fresh LoRA adapter on the 4 training folds.
2. Evaluate CSR on a sample from the held-out fold.

Final score = mean per-record CSR across folds, ∈ [0, 1].

In [ ]:
import random
from src.crllm.pipeline.lora_cv import load_records, run_kfold

records = load_records(AUG_JSONL)
print(f"Loaded {len(records):,} records from {AUG_JSONL.name}")

if SUBSAMPLE_N and len(records) > SUBSAMPLE_N:
    rng = random.Random(SEED)
    records = rng.sample(records, SUBSAMPLE_N)
    print(f"Subsampled to {len(records):,}")

In [ ]:
train_kwargs = dict(
    base_model              = BASE_MODEL,
    lora_rank               = LORA_RANK,
    learning_rate           = LEARNING_RATE,
    num_epochs              = NUM_EPOCHS,
    per_device_batch_size   = PER_DEVICE_BS,
    grad_accumulation_steps = GRAD_ACCUM_STEPS,
    max_seq_length          = MAX_SEQ_LENGTH,
)
eval_kwargs = dict(base_model=BASE_MODEL)

summary = run_kfold(
    records       = records,
    output_root   = CV_OUTPUT_DIR,
    k             = K_FOLDS,
    eval_per_fold = EVAL_PER_FOLD,
    seed          = SEED,
    train_kwargs  = train_kwargs,
    eval_kwargs   = eval_kwargs,
)

## 8. Final score & per-fold breakdown

In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt

print(f"\n{'='*55}")
print(f"  FINAL 5-FOLD LORA CSR = {summary['final_score']:.4f}")
print(f"{'='*55}")

df = pd.DataFrame(summary["fold_scores"])
print("\nPer-fold:")
print(df.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(df["fold"].astype(str), df["csr"], color="#2196F3", alpha=0.85)
ax.axhline(summary["final_score"], color="black", linestyle="--", label=f"mean={summary['final_score']:.3f}")
ax.set_xlabel("Fold"); ax.set_ylabel("CSR (∈ [0, 1])")
ax.set_title(f"5-Fold LoRA CV — Llama-3.2-1B on RECAST-30K\n(rank={LORA_RANK}, lr={LEARNING_RATE}, epochs={NUM_EPOCHS})")
ax.set_ylim(0, 1.0); ax.grid(axis="y", alpha=0.3); ax.legend()
plt.tight_layout()
plt.savefig(CV_OUTPUT_DIR / "kfold_csr.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nArtifacts: {CV_OUTPUT_DIR}/")
print("  summary.json       ← final_score + per-fold")
print("  all_results.json   ← per-record CSR")
print("  fold_<n>/lora_adapter/   ← trained adapter per fold")
print("  kfold_csr.png      ← bar plot")